# Phase 4 — TE Model: Tuned XGBoost with Walk-Forward Optuna Search

Trained only on TE's Phase 3-selected 9 features: `scarcity_z`, `air_yards_share`, `receiving_epa`, `wopr`, `target_share`, `draft_tier_Round 2-3`, `draft_pick_inverse`, `vorp_delta_yoy`, `injury_designations_count`. Structured identically to `06a_model_qb.ipynb` / `06b_model_rb.ipynb` / `06c_model_wr.ipynb`, with every lesson already proven applied from the start:

1. **`max_depth` 2-8 from the first run** — confirmed a real win for QB and RB, and the winning configuration for WR; no reason to re-litigate a third and fourth time.
2. **The `low_confidence_extreme_delta` flag and delta-bucket diagnostic are built in from the start.** `vorp_delta_yoy`'s extreme-tail weakness is now a 3-for-3 cross-position finding (QB 1.29x, RB 1.41x, WR 1.45x) — checked immediately for TE rather than assumed.
3. **The `scarcity_z` tail-behavior check is also built in from the start**, reported with both deciles shown separately rather than only an averaged number — WR found its top decile to be the hardest segment in the dataset once checked this way; RB's and QB's own versions of this check had not yet been redone with the same decile-separated method as of this notebook's first version. No assumption about which pattern TE will match.
4. **The monotonic-constraint diagnostic makes no assumption from any prior position** — QB said no, RB said yes, WR said no. TE decides on its own held-out folds.

TE also has two things no prior position notebook has had to deal with: a **categorical dummy feature** (`draft_tier_Round 2-3`, one-hot encoded from `draft_capital_tier`) and a feature with a **structurally missing season** (`injury_designations_count` is `NaN`, not `0`, for every 2008 row — `load_injuries()` has a hard floor at 2009). Both are handled explicitly below rather than silently.

In [1]:
from datetime import datetime

print(f"Results as of {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}, pulling live nflreadpy data -- "
      "rerunning this notebook will reflect any upstream corrections made to that data since.")

Results as of 2026-09-17 14:14 Central Daylight Time, pulling live nflreadpy data -- rerunning this notebook will reflect any upstream corrections made to that data since.


## Setup: load `features_df` and build TE's target rows

`air_yards_share`, `receiving_epa`, `wopr`, `target_share` come straight from `vorp_labels.parquet`'s seasonal stats, same as WR. `scarcity_z`, `vorp_delta_yoy`, `draft_pick_inverse` are built the same way every prior position has built them. Two features are new to a `06x` notebook:

- **`draft_tier_Round 2-3`**: built directly as a 0/1 indicator (`draft_round.isin([2, 3])`), not via `pd.get_dummies` on the full `draft_capital_tier` column — only this one tier survived Phase 3's RFECV for TE, so the other three tiers (`Round 1`, `Round 4+`, `UDFA`) are never constructed as columns at all.
- **`injury_designations_count`**: a concurrent-season count of Questionable/Doubtful/Out designations, built exactly as Phase 3 documented it (`04_feature_creation.ipynb`, Section 3b). **2008 rows are left `NaN`, not filled with 0** — `load_injuries()` has no 2008 data at all, so a `NaN` here means "unknown," not "healthy." Every 2009+ real zero is filled in as a genuine 0. This `NaN` is **not** imputed or dropped: `XGBRegressor`'s native missing-value handling (it learns an optimal default split direction for missing values during training, the same mechanism Phase 3's own untuned RFECV run relied on for this exact column) handles it directly, so 2008 rows stay in the training data rather than being silently discarded.

In [2]:
import sys
from pathlib import Path

import nflreadpy as nfl
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FEATURES = ["scarcity_z", "air_yards_share", "receiving_epa", "wopr", "target_share",
            "draft_tier_Round 2-3", "draft_pick_inverse", "vorp_delta_yoy", "injury_designations_count"]

vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")
te = vorp_labels[vorp_labels["position"] == "TE"][
    ["season", "player_id", "player_display_name", "vorp", "vorp_next",
     "wopr", "receiving_epa", "air_yards_share", "target_share"]
].copy()

# scarcity_z (within-position, within-season standardization -- same recipe as Phase 3)
season_position_stats = (
    vorp_labels.groupby(["season", "position"])["vorp"]
    .agg(position_mean_vorp="mean", position_std_vorp_that_season="std")
    .reset_index()
)
te_stats = season_position_stats[season_position_stats["position"] == "TE"]
te = te.merge(te_stats[["season", "position_mean_vorp", "position_std_vorp_that_season"]], on="season", how="left")
te["scarcity_z"] = (te["vorp"] - te["position_mean_vorp"]) / te["position_std_vorp_that_season"]
te = te.drop(columns=["position_mean_vorp", "position_std_vorp_that_season"])

# vorp_delta_yoy
prior = te[["player_id", "season", "vorp"]].copy()
prior["season"] = prior["season"] + 1
prior = prior.rename(columns={"vorp": "vorp_last_season"})
te = te.merge(prior, on=["player_id", "season"], how="left")
te["vorp_delta_yoy"] = te["vorp"] - te["vorp_last_season"]
te = te.drop(columns=["vorp_last_season"])

# draft_pick_inverse, draft_tier_Round 2-3
players = nfl.load_players().to_pandas()
te = te.merge(players[["gsis_id", "draft_round", "draft_pick"]], left_on="player_id", right_on="gsis_id", how="left")
te = te.drop(columns=["gsis_id"])
te["draft_pick_inverse"] = 1 / te["draft_pick"]
te["draft_tier_Round 2-3"] = te["draft_round"].isin([2, 3]).astype(float)
te = te.drop(columns=["draft_round", "draft_pick"])

# injury_designations_count (concurrent-season; 2008 stays NaN -- no load_injuries() data that year)
injuries = nfl.load_injuries(seasons=True).to_pandas()
flagged = injuries[injuries["report_status"].isin(["Questionable", "Doubtful", "Out"])]
injury_counts = flagged.groupby(["gsis_id", "season"]).size().reset_index(name="injury_designations_count")
te = te.merge(injury_counts, left_on=["player_id", "season"], right_on=["gsis_id", "season"], how="left")
te = te.drop(columns=["gsis_id"])
has_2009plus = te["season"] >= 2009
te.loc[has_2009plus, "injury_designations_count"] = te.loc[has_2009plus, "injury_designations_count"].fillna(0)

# Keep 2025's rows before dropping them below -- their vorp_next is 2026's outcome, which
# doesn't exist yet, so dropna() would discard them. Needed later for the live 2026 inference section.
te_2025_features = te[te["season"] == 2025].copy()

te = te.dropna(subset=["vorp_next"]).reset_index(drop=True)
print(f"TE rows with a usable label: {len(te)}, seasons {te['season'].min()}-{te['season'].max()}")
print(te[FEATURES].isna().mean().rename("null_rate"))
print("\ninjury_designations_count null rate by season (should be 100% for 2008 only, ~0% elsewhere):")
print(te.groupby("season")["injury_designations_count"].apply(lambda s: s.isna().mean()).to_string())

C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TE rows with a usable label: 1545, seasons 2008-2024
scarcity_z                   0.000000
air_yards_share              0.000000
receiving_epa                0.031715
wopr                         0.030421
target_share                 0.030421
draft_tier_Round 2-3         0.000000
draft_pick_inverse           0.256311
vorp_delta_yoy               0.282201
injury_designations_count    0.053722
Name: null_rate, dtype: float64

injury_designations_count null rate by season (should be 100% for 2008 only, ~0% elsewhere):
season
2008    1.0
2009    0.0
2010    0.0
2011    0.0
2012    0.0
2013    0.0
2014    0.0
2015    0.0
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    0.0


## Walk-forward folds (same structure as every prior position)

In [3]:
MIN_TRAIN_SEASONS = 9


def make_walk_forward_folds(df, min_train_seasons=MIN_TRAIN_SEASONS):
    seasons_sorted = sorted(df["season"].unique())
    folds = []
    for i in range(min_train_seasons, len(seasons_sorted)):
        train_seasons = sorted(seasons_sorted[:i])
        test_season = seasons_sorted[i]
        train_idx = df.index[df["season"].isin(train_seasons)].to_numpy()
        test_idx = df.index[df["season"] == test_season].to_numpy()
        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append((train_idx, test_idx, test_season, train_seasons))
    return folds


folds = make_walk_forward_folds(te)
print(f"{len(folds)} walk-forward folds, test seasons: {[f[2] for f in folds]}")

8 walk-forward folds, test seasons: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## Monotonic constraints: checked per feature, not assumed

Each of TE's 9 features checked individually — no assumption carried over from QB, RB, or WR's own answers:

| Feature | Constraint | Reasoning |
|---|---|---|
| `scarcity_z` | **+1** | Standing further above the field *this* season should never predict a *worse* expected standing next season, all else equal. Unambiguous — same reasoning as every prior position. |
| `air_yards_share` | **+1** | Direct opportunity metric, same reasoning as WR's notebook — no realistic mechanism where a larger share of the team's downfield volume this season predicts *lower* value next season, holding the rest of the feature set fixed. |
| `receiving_epa` | **+1** | Direct efficiency metric, same reasoning QB's notebook applied to `passing_epa` and WR's applied to `receiving_epa` itself. |
| `wopr` | **+1** | Direct, designed-for-this opportunity metric, same reasoning as WR. |
| `target_share` | **+1** | Same opportunity-metric logic as `wopr`/`air_yards_share`. |
| `draft_tier_Round 2-3` | **0 (none)** | A fixed pre-career proxy for organizational investment, not a performance metric — same real ambiguity as `draft_pick_inverse` at every prior position. Phase 3 confirmed this is a real, well-populated signal for TE (97 distinct players), but a real *signal* is not the same as an unambiguous *direction*: conditional on the other 8 features already carrying realized performance and opportunity, whether Round 2-3 pedigree specifically should point the prediction up or down isn't obvious enough to force. Left unconstrained rather than guessed. |
| `draft_pick_inverse` | **0 (none)** | Same real ambiguity as every prior position — a fixed pre-career proxy, not a performance metric. |
| `vorp_delta_yoy` | **+1** | An improving trajectory is more plausible to continue than to reverse, especially once `scarcity_z` already anchors the current level. Same reasoning as every prior position. |
| `injury_designations_count` | **0 (none)** | The most genuinely ambiguous feature in this set, and new to any `06x` notebook. The intuitive case for `-1` (more Q/D/O tags this season → worse health outlook → lower value next season) is real, but there's a real confound working the other way: an established, high-usage TE can rack up multiple minor "wear and tear" designations *because* he's the focal option who plays through niggles rather than sitting, not because he's declining — the same kind of role/usage confound that already kept `age` and `draft_pick_inverse` unconstrained everywhere else. No unambiguous mechanism, so left free. |

**6 of 9 features get a constraint** (`scarcity_z`, `air_yards_share`, `receiving_epa`, `wopr`, `target_share`, `vorp_delta_yoy`, all `+1`); `draft_tier_Round 2-3`, `draft_pick_inverse`, and `injury_designations_count` are left free.

In [4]:
MONOTONE_CONSTRAINTS = (1, 1, 1, 1, 1, 0, 0, 1, 0)  # matches FEATURES order exactly
print(dict(zip(FEATURES, MONOTONE_CONSTRAINTS)))

{'scarcity_z': 1, 'air_yards_share': 1, 'receiving_epa': 1, 'wopr': 1, 'target_share': 1, 'draft_tier_Round 2-3': 0, 'draft_pick_inverse': 0, 'vorp_delta_yoy': 1, 'injury_designations_count': 0}


## Per-fold Optuna search (TPE sampler, pruning enabled)

Same search space as every prior position: `max_depth` (2-8, wide from the start), `min_child_weight` (1-10), `reg_lambda` (log-scale 0.1-10), `learning_rate` (log-scale 0.01-0.3), `subsample` (0.6-1.0), 40 trials/fold, early stopping (up to 500 rounds, 20-round patience) instead of tuning `n_estimators` directly. The inner-fold validation split, `MedianPruner`, and `XGBRegressor`-first interface are all unchanged from QB/RB/WR. Out-of-fold predictions (plus each row's `vorp_delta_yoy` and `scarcity_z`) are collected during the search, needed for the delta-bucket and `scarcity_z` diagnostics below.

In [5]:
N_TRIALS = 40


class OptunaPruningCallback(xgb.callback.TrainingCallback):
    """Reports each boosting round's validation MAE back to Optuna so its
    pruner can stop a clearly-unpromising trial mid-training, not just
    compare finished trials against each other after the fact.

    Reads "validation_0" -- XGBRegressor's auto-generated eval_set name --
    not the "valid" key the native xgb.train(evals=[...]) API would use."""

    def __init__(self, trial):
        self.trial = trial

    def after_iteration(self, model, epoch, evals_log):
        score = evals_log["validation_0"]["mae"][-1]
        self.trial.report(score, step=epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()
        return False


def run_optuna_for_fold(df, train_idx, monotone_constraints, n_trials=N_TRIALS, seed=42):
    train_seasons_sorted = sorted(df.loc[train_idx, "season"].unique())
    inner_valid_season = train_seasons_sorted[-1]
    inner_train_seasons = train_seasons_sorted[:-1]
    inner_train_idx = df.index[df["season"].isin(inner_train_seasons) & df.index.isin(train_idx)]
    inner_valid_idx = df.index[(df["season"] == inner_valid_season) & df.index.isin(train_idx)]

    X_train = df.loc[inner_train_idx, FEATURES]
    y_train = df.loc[inner_train_idx, "vorp_next"]
    X_valid = df.loc[inner_valid_idx, FEATURES]
    y_valid = df.loc[inner_valid_idx, "vorp_next"]

    def objective(trial):
        params = {
            "objective": "reg:squarederror", "eval_metric": "mae",
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10, log=True),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "monotone_constraints": monotone_constraints, "seed": seed,
            "n_estimators": 500, "early_stopping_rounds": 20,
            "callbacks": [OptunaPruningCallback(trial)],
        }
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
        trial.set_user_attr("best_iteration", model.best_iteration)
        return model.best_score

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study


def fold_gain_share(model):
    """Gain importance for one fold's refit booster, normalized to shares
    over FEATURES (0 for any feature the booster never split on) -- same
    definition Phase 3 used (mean gain share across walk-forward folds)."""
    scores = model.get_booster().get_score(importance_type="gain")
    raw = np.array([scores.get(f, 0.0) for f in FEATURES])
    total = raw.sum()
    return raw / total if total > 0 else raw


def run_all_folds(df, folds, monotone_constraints, label):
    fold_results, fold_best_params, fold_gain_shares, oof_rows = [], [], [], []
    for tr, te_idx, test_season, train_seasons in folds:
        study = run_optuna_for_fold(df, tr, monotone_constraints)
        best_params = dict(study.best_params)
        best_iteration = study.best_trial.user_attrs["best_iteration"]

        X_train_full = df.loc[tr, FEATURES]
        y_train_full = df.loc[tr, "vorp_next"]
        X_test = df.loc[te_idx, FEATURES]
        final_model = xgb.XGBRegressor(
            objective="reg:squarederror", monotone_constraints=monotone_constraints, seed=42,
            n_estimators=max(best_iteration, 1), **best_params,
        )
        final_model.fit(X_train_full, y_train_full)
        preds = final_model.predict(X_test)
        actual = df.loc[te_idx, "vorp_next"]

        mae = mean_absolute_error(actual, preds)
        rmse = mean_squared_error(actual, preds) ** 0.5
        rho = spearmanr(actual, preds)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan

        fold_results.append({"position": "TE", "test_season": test_season, "n_test": len(te_idx), "mae": mae, "rmse": rmse, "spearman": rho})
        fold_best_params.append(best_params | {"n_estimators": best_iteration})
        fold_gain_shares.append(fold_gain_share(final_model))
        oof_rows.append(pd.DataFrame({
            "test_season": test_season,
            "predicted_vorp_next": preds,
            "actual_vorp_next": actual.to_numpy(),
            "vorp_delta_yoy": df.loc[te_idx, "vorp_delta_yoy"].to_numpy(),
            "scarcity_z": df.loc[te_idx, "scarcity_z"].to_numpy(),
        }))

    fold_df = pd.DataFrame(fold_results)
    mean_gain_df = pd.DataFrame({"feature": FEATURES, "mean_gain_share": np.mean(fold_gain_shares, axis=0)})
    oof_df = pd.concat(oof_rows, ignore_index=True)
    print(f"[{label}] Mean MAE: {fold_df['mae'].mean():.2f}, Mean RMSE: {fold_df['rmse'].mean():.2f}, Mean Spearman: {fold_df['spearman'].mean():.3f}")
    return fold_df, pd.DataFrame(fold_best_params), mean_gain_df, oof_df


fold_metrics_constrained_df, fold_params_constrained_df, gain_constrained_df, oof_constrained_df = run_all_folds(te, folds, MONOTONE_CONSTRAINTS, label="tuned + monotonic")
print(fold_metrics_constrained_df.to_string(index=False))

[tuned + monotonic] Mean MAE: 26.33, Mean RMSE: 35.36, Mean Spearman: 0.689
position  test_season  n_test       mae      rmse  spearman
      TE         2017      88 26.471175 38.615664  0.623827
      TE         2018      91 27.228287 38.919575  0.543816
      TE         2019      95 29.219739 36.863837  0.645910
      TE         2020      97 29.130949 35.976304  0.710738
      TE         2021      99 23.004669 30.919756  0.768971
      TE         2022      95 23.452870 31.475287  0.753389
      TE         2023      95 26.678266 37.750341  0.683005
      TE         2024     106 25.491479 32.326588  0.779856


## Diagnostic: does the monotonic constraint actually help?

No assumption carried over from any prior position — checked directly on TE's own held-out folds.

In [6]:
NO_CONSTRAINTS = (0, 0, 0, 0, 0, 0, 0, 0, 0)
fold_metrics_unconstrained_df, fold_params_unconstrained_df, gain_unconstrained_df, oof_unconstrained_df = run_all_folds(te, folds, NO_CONSTRAINTS, label="tuned, no constraints")

[tuned, no constraints] Mean MAE: 26.59, Mean RMSE: 35.87, Mean Spearman: 0.691


**Finding: TE is the first genuine tie among the four positions checked.** Constrained wins MAE (26.33 vs. 26.59, ~1.0% better); unconstrained wins Spearman (0.691 vs. 0.689), but by a margin (0.002) that's well within fold-to-fold noise for 8 folds of ~90-100 rows each — nowhere near the size of QB's, RB's, or WR's own clean double-wins. Neither configuration wins both metrics, unlike every prior position.

**Tie-break, stated explicitly rather than silently picked**: MAE is treated as decisive here because it's the metric this notebook's own tuning objective directly optimizes (`run_optuna_for_fold`'s `objective()` returns `model.best_score`, the early-stopped validation MAE) — when the two metrics disagree and the Spearman gap is this small, the metric actually being searched over is the more meaningful tiebreaker. **The constrained model is official for TE.** This is a materially different situation from QB (unconstrained, clean), RB (constrained, clean), and WR (unconstrained, clean) — TE doesn't extend either "trend," it's simply its own close call.

## Naive baseline, same folds — apples-to-apples with Phase 3

In [7]:
naive_records = []
for tr, te_idx, test_season, train_seasons in folds:
    actual = te.loc[te_idx, "vorp_next"]
    naive_pred = te.loc[te_idx, "vorp"]
    mae = mean_absolute_error(actual, naive_pred)
    rho = spearmanr(actual, naive_pred)[0] if len(actual) >= 2 and actual.nunique() > 1 else np.nan
    naive_records.append({"test_season": test_season, "mae": mae, "spearman": rho})
naive_df = pd.DataFrame(naive_records)
naive_mae, naive_spearman = naive_df["mae"].mean(), naive_df["spearman"].mean()
print(f"Naive baseline -- Mean MAE: {naive_mae:.2f}, Mean Spearman: {naive_spearman:.3f}")

tuned_mae, tuned_spearman = fold_metrics_constrained_df["mae"].mean(), fold_metrics_constrained_df["spearman"].mean()
comparison = pd.DataFrame([
    {"model": "Naive (this season's VORP)", "mae": round(naive_mae, 2), "spearman": round(naive_spearman, 3)},
    {"model": "Phase 4 tuned, no constraints (diagnostic only)", "mae": round(fold_metrics_unconstrained_df['mae'].mean(), 2), "spearman": round(fold_metrics_unconstrained_df['spearman'].mean(), 3)},
    {"model": "Phase 4 tuned + monotonic (official, shipped)", "mae": round(tuned_mae, 2), "spearman": round(tuned_spearman, 3)},
])
print(comparison.to_string(index=False))
print(f"\nTuned (constrained) vs naive: {(naive_mae - tuned_mae) / naive_mae:.1%} MAE change, {tuned_spearman - naive_spearman:+.3f} Spearman change")

Naive baseline -- Mean MAE: 27.28, Mean Spearman: 0.691
                                          model   mae  spearman
                     Naive (this season's VORP) 27.28     0.691
Phase 4 tuned, no constraints (diagnostic only) 26.59     0.691
  Phase 4 tuned + monotonic (official, shipped) 26.33     0.689

Tuned (constrained) vs naive: 3.5% MAE change, -0.002 Spearman change


**Finding**: unlike every prior position, TE's naive baseline is essentially **not beaten on Spearman** — 0.689 tuned vs. 0.691 naive (a tiny -0.002 give-back), while MAE improves modestly (27.28 → 26.33, 3.5%). This matches Phase 3's own untuned-era finding that TE showed "small, real, same-direction improvement" on both metrics — the tuned model preserves that MAE edge but no longer clears naive on ranking quality by even the small margin Phase 3 found untuned. Worth reading plainly: TE's real edge over "assume next season repeats this season" is on **average error size**, not on **who ends up ranked above whom** — a materially weaker claim than QB/RB/WR could each make for their own official models.

## Delta-bucket MAE diagnostic: does TE continue the accelerating cross-position trend?

Confirmed for QB (1.29x), RB (1.41x), and WR (1.45x) — a monotonically *increasing* degradation ratio across the three positions checked so far. TE is checked here the same way, using out-of-fold predictions from the official (constrained) model pooled across all 8 folds.

In [8]:
oof_df = oof_constrained_df.copy()
oof_df["abs_delta"] = oof_df["vorp_delta_yoy"].abs()
oof_df["abs_error"] = (oof_df["predicted_vorp_next"] - oof_df["actual_vorp_next"]).abs()

DELTA_BUCKET_THRESHOLD = 100
le_bucket = oof_df[oof_df["abs_delta"] <= DELTA_BUCKET_THRESHOLD]
gt_bucket = oof_df[oof_df["abs_delta"] > DELTA_BUCKET_THRESHOLD]

le_mae, gt_mae = le_bucket["abs_error"].mean(), gt_bucket["abs_error"].mean()
overall_mae_pooled = oof_df["abs_error"].mean()

print(f"Overall held-out MAE (pooled out-of-fold): {overall_mae_pooled:.2f} "
      f"(vs. {fold_metrics_constrained_df['mae'].mean():.2f}, the unweighted mean-of-fold-means above)")
print(f"|vorp_delta_yoy| <= {DELTA_BUCKET_THRESHOLD}: n={len(le_bucket)}, MAE={le_mae:.2f}")
print(f"|vorp_delta_yoy| > {DELTA_BUCKET_THRESHOLD}:  n={len(gt_bucket)}, MAE={gt_mae:.2f}")
print(f"Extreme bucket is {gt_mae / le_mae:.2f}x worse than the non-extreme bucket")

oof_df["delta_quartile"] = pd.qcut(oof_df["abs_delta"], 4, labels=["Q1 (smallest)", "Q2", "Q3", "Q4 (largest)"])
quartile_summary = oof_df.groupby("delta_quartile", observed=True).agg(
    n=("abs_error", "size"), mae=("abs_error", "mean"),
    delta_range=("abs_delta", lambda x: f"{x.min():.1f}-{x.max():.1f}"),
)
print("\nQuartile breakdown of |vorp_delta_yoy| (out-of-fold MAE by quartile):")
print(quartile_summary.to_string())

print("\nCross-position degradation ratio so far:")
print(pd.DataFrame([
    {"position": "QB", "ratio": 1.29}, {"position": "RB", "ratio": 1.41},
    {"position": "WR", "ratio": 1.45}, {"position": "TE", "ratio": round(gt_mae / le_mae, 2)},
]).to_string(index=False))

Overall held-out MAE (pooled out-of-fold): 26.31 (vs. 26.33, the unweighted mean-of-fold-means above)
|vorp_delta_yoy| <= 100: n=569, MAE=26.83
|vorp_delta_yoy| > 100:  n=18, MAE=44.43
Extreme bucket is 1.66x worse than the non-extreme bucket

Quartile breakdown of |vorp_delta_yoy| (out-of-fold MAE by quartile):
                  n        mae delta_range
delta_quartile                            
Q1 (smallest)   147  20.763667     0.1-8.4
Q2              147  21.666946    8.4-22.1
Q3              146  30.790880   22.2-40.4
Q4 (largest)    147  36.295242  40.6-165.6

Cross-position degradation ratio so far:
position  ratio
      QB   1.29
      RB   1.41
      WR   1.45
      TE   1.66


**Finding: the trend doesn't just continue, it accelerates.** TE's extreme-tail degradation is **1.66x** — the largest gap of the four positions checked (QB 1.29x → RB 1.41x → WR 1.45x → TE 1.66x), and a bigger jump than any single step before it. The quartile breakdown is monotonically increasing (20.76 → 21.67 → 30.79 → 36.30), with the steepest single jump landing between Q2 and Q3 rather than spread evenly — but the overall direction is unambiguous. Four for four positions now confirm this pattern, and TE's version of it is the sharpest yet.

## `scarcity_z` tail-behavior check — reported by decile, not averaged

No assumption about which prior pattern TE will match: WR found its top `scarcity_z` decile to be the *hardest* segment in its whole dataset once checked by decile rather than averaged; RB's and QB's own claims of "both extremes fit better" had, at the time this notebook was first written, never actually been checked the same decile-separated way — they came from an averaged number computed outside any notebook. Both deciles are reported separately below, not just averaged together — averaging is exactly what would have hidden WR's own asymmetry (and, as later confirmed in `06b_model_rb.ipynb`, RB's too).

In [9]:
oof_df["scarcity_decile"] = pd.qcut(oof_df["scarcity_z"], 10, labels=False, duplicates="drop")
top_decile = oof_df[oof_df["scarcity_decile"] == oof_df["scarcity_decile"].max()]
bottom_decile = oof_df[oof_df["scarcity_decile"] == 0]
middle_80 = oof_df[~oof_df["scarcity_decile"].isin([0, oof_df["scarcity_decile"].max()])]

print(f"scarcity_z top decile (best TEs):        n={len(top_decile)}, MAE={top_decile['abs_error'].mean():.2f}, "
      f"z range [{top_decile['scarcity_z'].min():.2f}, {top_decile['scarcity_z'].max():.2f}]")
print(f"scarcity_z bottom decile (replacement level): n={len(bottom_decile)}, MAE={bottom_decile['abs_error'].mean():.2f}, "
      f"z range [{bottom_decile['scarcity_z'].min():.2f}, {bottom_decile['scarcity_z'].max():.2f}]")
print(f"scarcity_z middle 80%:                   n={len(middle_80)}, MAE={middle_80['abs_error'].mean():.2f}")

combined_extreme = pd.concat([top_decile, bottom_decile])
print(f"\nBoth extreme deciles combined: n={len(combined_extreme)}, MAE={combined_extreme['abs_error'].mean():.2f} "
      f"(vs. overall pooled held-out {overall_mae_pooled:.2f}, vs. middle-80% {middle_80['abs_error'].mean():.2f})")

scarcity_z top decile (best TEs):        n=77, MAE=44.82, z range [1.65, 4.79]
scarcity_z bottom decile (replacement level): n=77, MAE=14.97, z range [-0.91, -0.82]
scarcity_z middle 80%:                   n=612, MAE=25.40

Both extreme deciles combined: n=154, MAE=29.90 (vs. overall pooled held-out 26.31, vs. middle-80% 25.40)


**Finding: TE joins WR.** The bottom decile (replacement-level TEs) is easy — MAE 14.97, well below the 25.40 middle-80% and the overall held-out MAE — consistent with the "low-variance, near-floor outcomes are simply easier" logic every position has shown. But the **top decile (the actual TE1s) is the hardest segment in the entire dataset at MAE 44.82** — worse than even the `vorp_delta_yoy` extreme-tail bucket (44.43) directly above. Averaged together the two extremes land at 29.90, moderately worse than the overall MAE, not "comfortably better" the way an averaged-only read would suggest.

**Update, added once RB's own version of this check existed**: `06b_model_rb.ipynb` was later given the identical decile-separated check (it had none before) and found the same top-decile-is-hardest pattern — RB's own combined-extreme average had also looked roughly neutral and hidden the same asymmetry underneath. That makes this **3 of 3 positions actually checked this way** (WR, TE, and now RB) showing "the top decile is the hardest segment," with QB's own claim still unverified by a real decile-separated check in `06a_model_qb.ipynb` as of this writing. The originally-suspected "receiving-volume positions vs. run/pocket-presence positions" split does not hold up now that RB has been checked properly — this looks more like a universal pattern across positions than a family-level one, pending QB's own real check.

## `low_confidence_extreme_delta` flag — built in from the start

In [10]:
LOW_CONFIDENCE_DELTA_THRESHOLD = 100  # matches the |vorp_delta_yoy| bucket confirmed above


def add_low_confidence_flag(df, delta_col="vorp_delta_yoy"):
    """True wherever |vorp_delta_yoy| exceeds the threshold this notebook's own delta-bucket
    diagnostic (above) found to be a real, measured weak spot -- same flag-not-drop pattern as
    Phase 2's low_snap_next_season and QB/RB/WR's low_confidence_extreme_delta. Informational
    only: this model was never retrained or refit with this flag as a feature or filter.
    NaN (a player with no computable prior-season delta, e.g. a true rookie) evaluates to
    False here, NOT True -- that is a real gap in what this flag can tell you, not a judgment
    that a missing-history row is reliable. Checked explicitly in the live 2026 section below."""
    return df[delta_col].abs() > LOW_CONFIDENCE_DELTA_THRESHOLD


n_flagged_train = add_low_confidence_flag(te).sum()
n_nan_delta = te["vorp_delta_yoy"].isna().sum()
print(f"low_confidence_extreme_delta threshold: |vorp_delta_yoy| > {LOW_CONFIDENCE_DELTA_THRESHOLD}")
print(f"Training rows flagged: {n_flagged_train} / {len(te)} ({n_flagged_train / len(te):.1%})")
print(f"Training rows with no computable delta at all (NaN, e.g. no usable prior season): {n_nan_delta} / {len(te)} ({n_nan_delta / len(te):.1%})")

low_confidence_extreme_delta threshold: |vorp_delta_yoy| > 100
Training rows flagged: 41 / 1545 (2.7%)
Training rows with no computable delta at all (NaN, e.g. no usable prior season): 436 / 1545 (28.2%)


## Save fold-by-fold results

Saves the **constrained** (official, winning-on-tiebreak) fold metrics.

In [11]:
out_path = REPO_ROOT / "data" / "processed" / "fold_metrics_te.csv"
fold_metrics_constrained_df.to_csv(out_path, index=False)
print(f"Saved {len(fold_metrics_constrained_df)} fold rows to {out_path}")

Saved 8 fold rows to C:\Users\viraj\OneDrive\Desktop\ML Trial\ML-Test\data\processed\fold_metrics_te.csv


## Final model: trained on all available data, using the *stable* hyperparameter region

Median hyperparameters across all 8 folds' best trials from the **constrained** search (the official, winning-on-tiebreak configuration), same principled `n_estimators` selection as every prior position.

In [12]:
stable_params = {
    "max_depth": int(round(fold_params_constrained_df["max_depth"].median())),
    "min_child_weight": int(round(fold_params_constrained_df["min_child_weight"].median())),
    "reg_lambda": float(fold_params_constrained_df["reg_lambda"].median()),
    "learning_rate": float(fold_params_constrained_df["learning_rate"].median()),
    "subsample": float(fold_params_constrained_df["subsample"].median()),
}
print("Per-fold best hyperparameters (constrained search):")
print(fold_params_constrained_df.drop(columns=["n_estimators"]).to_string(index=False))
print("\nStable (median) hyperparameters chosen for the final model:")
print(stable_params)

all_seasons_sorted = sorted(te["season"].unique())
final_valid_season = all_seasons_sorted[-1]
final_train_seasons = all_seasons_sorted[:-1]

final_train_idx = te.index[te["season"].isin(final_train_seasons)]
final_valid_idx = te.index[te["season"] == final_valid_season]

X_final_train = te.loc[final_train_idx, FEATURES]
y_final_train = te.loc[final_train_idx, "vorp_next"]
X_final_valid = te.loc[final_valid_idx, FEATURES]
y_final_valid = te.loc[final_valid_idx, "vorp_next"]

probe_model = xgb.XGBRegressor(
    objective="reg:squarederror", eval_metric="mae", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=500, early_stopping_rounds=20, **stable_params,
)
probe_model.fit(X_final_train, y_final_train, eval_set=[(X_final_valid, y_final_valid)], verbose=False)
final_n_estimators = max(probe_model.best_iteration, 1)
print(f"\nFinal n_estimators (early-stopped against {final_valid_season}): {final_n_estimators}")

# Refit on ALL labeled data, no held-out slice left over, using the fixed iteration count.
final_model = xgb.XGBRegressor(
    objective="reg:squarederror", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=final_n_estimators, **stable_params,
)
final_model.fit(te[FEATURES], te["vorp_next"])

models_dir = REPO_ROOT / "data" / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "te_model.json"
final_model.save_model(str(model_path))
print(f"Final model trained on {len(te)} rows (seasons {te['season'].min()}-{te['season'].max()}), saved to {model_path}")

Per-fold best hyperparameters (constrained search):
 max_depth  min_child_weight  reg_lambda  learning_rate  subsample
         3                 3    7.726082       0.265707   0.940964
         4                 6    0.114735       0.279763   0.687486
         3                 3    0.157377       0.150368   0.846819
         2                10    0.300804       0.260553   0.672199
         8                 9    0.125508       0.130516   0.696494
         2                 5    0.328085       0.282433   0.786335
         3                 5    0.116157       0.267886   0.618047
         2                10    2.841753       0.232673   0.644155

Stable (median) hyperparameters chosen for the final model:
{'max_depth': 3, 'min_child_weight': 6, 'reg_lambda': 0.22909039014724497, 'learning_rate': 0.2631297146487238, 'subsample': 0.6919899680068478}

Final n_estimators (early-stopped against 2024): 29
Final model trained on 1545 rows (seasons 2008-2024), saved to C:\Users\viraj\OneDrive

## Does the final model still make football sense? And does `draft_tier_Round 2-3` survive real tuning?

Phase 3's untuned RFECV run found `draft_tier_Round 2-3` carrying real, meaningful gain for TE (ranked 6th of 10 candidate features, 0.0448 mean gain share) — confirmed at the time as genuine signal, not a small-sample fluke like K's version of the same dummy (K: only 5 distinct kickers ever drafted Round 2-3; TE: 97 distinct players, ~24% of all TE rows). The real question for this notebook: does that finding survive actual walk-forward Optuna tuning, the same "does the earlier finding hold up under real tuning" check that TD-rate regression *failed* for QB (`05b_qb_feature_experiment.ipynb`)?

In [13]:
gain_scores = final_model.get_booster().get_score(importance_type="gain")
importance_df = (
    pd.DataFrame({"feature": list(gain_scores.keys()), "gain": list(gain_scores.values())})
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
missing = [f for f in FEATURES if f not in importance_df["feature"].values]
if missing:
    importance_df = pd.concat([importance_df, pd.DataFrame({"feature": missing, "gain": 0.0})], ignore_index=True)
importance_df["gain_share"] = importance_df["gain"] / importance_df["gain"].sum()
print("Final model (constrained, official) gain importance:")
print(importance_df.to_string(index=False))

rank = importance_df.reset_index().set_index("feature")["index"] + 1
print(f"\ndraft_tier_Round 2-3 final rank: {rank['draft_tier_Round 2-3']} of {len(importance_df)}")

print("\ndraft_tier_Round 2-3 across every configuration checked:")
phase3_untuned = 0.044782
print(pd.DataFrame([
    {"config": "Phase 3 untuned RFECV (10-feature pool)", "mean_gain_share": phase3_untuned, "rank": "6th of 10"},
    {"config": "Phase 4 tuned + monotonic (official, per-fold mean)", "mean_gain_share": gain_constrained_df.set_index("feature").loc["draft_tier_Round 2-3", "mean_gain_share"], "rank": "6th of 9"},
    {"config": "Phase 4 tuned, no constraints (diagnostic, per-fold mean)", "mean_gain_share": gain_unconstrained_df.set_index("feature").loc["draft_tier_Round 2-3", "mean_gain_share"], "rank": "9th of 9 (last)"},
]).to_string(index=False))

Final model (constrained, official) gain importance:
                  feature          gain  gain_share
               scarcity_z 217644.859375    0.681725
             target_share  31571.474609    0.098891
            receiving_epa  18358.796875    0.057505
          air_yards_share  13822.817383    0.043297
                     wopr  13281.030273    0.041600
     draft_tier_Round 2-3   9048.652344    0.028343
       draft_pick_inverse   5500.573242    0.017229
           vorp_delta_yoy   5151.892578    0.016137
injury_designations_count   4875.873047    0.015273

draft_tier_Round 2-3 final rank: 6 of 9

draft_tier_Round 2-3 across every configuration checked:
                                                   config  mean_gain_share            rank
                  Phase 3 untuned RFECV (10-feature pool)         0.044782       6th of 10
      Phase 4 tuned + monotonic (official, per-fold mean)         0.028432        6th of 9
Phase 4 tuned, no constraints (diagnostic, per-fold mea

**Finding: `draft_tier_Round 2-3` survives real tuning — but only in the configuration that actually shipped.** In the **official constrained model**, it holds essentially the same rank Phase 3's untuned RFECV run found (6th of 9, mean gain share 0.028 vs. Phase 3's 0.045 in a 10-feature pool) — a genuine survival, not a coincidence of one favorable run. But under the **unconstrained diagnostic** (the alternative this notebook considered and rejected only on a narrow MAE tiebreak), it drops to **dead last** of all 9 features. This is a real, if secondary, instability: unlike TD-rate regression's clean rejection for QB (RFECV rejected it outright, in every configuration tried), `draft_tier_Round 2-3`'s survival is conditional on which constraint choice wins — and TE's constraint choice was the closest call of any position so far. Reported plainly rather than picking whichever number tells the cleaner story: the feature is real and does carry through to the shipped model, but it is measurably more fragile here than QB's `draft_pick_inverse` or RB's `age` ever were under their own wider searches.

**Football sense check otherwise**: `scarcity_z` dominates as always (68.2%), `target_share` is a clear #2 (9.9%) — TE again shows the WR-like pattern of usage features surviving with real weight, not RB's near-total elimination of the opportunity trio. No feature returns zero gain.

## Real predictions vs. reality — 2024 test season

Same check as every prior position: the 2024 walk-forward fold trains on 2008-2023 and predicts each TE's realized 2025 outcome from their real 2024 season. This reproduces the exact 2024 row already averaged into the constrained fold table above, broken out player-by-player.

In [14]:
last_fold_tr, last_fold_te, last_test_season, last_train_seasons = folds[-1]
assert last_test_season == 2024

last_fold_params = dict(fold_params_constrained_df.iloc[-1])
last_fold_n_estimators = int(last_fold_params.pop("n_estimators"))
# .iloc[-1] on a mixed-dtype DataFrame upcasts the whole row to float64 (e.g. max_depth -> 3.0),
# which XGBoost's C API rejects for integer params -- cast them back explicitly.
last_fold_params["max_depth"] = int(last_fold_params["max_depth"])
last_fold_params["min_child_weight"] = int(last_fold_params["min_child_weight"])

X_train_2024 = te.loc[last_fold_tr, FEATURES]
y_train_2024 = te.loc[last_fold_tr, "vorp_next"]
X_test_2024 = te.loc[last_fold_te, FEATURES]
y_test_2024 = te.loc[last_fold_te, "vorp_next"]

reproduced_2024_model = xgb.XGBRegressor(
    objective="reg:squarederror", monotone_constraints=MONOTONE_CONSTRAINTS, seed=42,
    n_estimators=last_fold_n_estimators, **last_fold_params,
)
reproduced_2024_model.fit(X_train_2024, y_train_2024)
predicted_2024 = reproduced_2024_model.predict(X_test_2024)

results_2024 = pd.DataFrame({
    "player": te.loc[last_fold_te, "player_display_name"].to_numpy(),
    "predicted_vorp_next": predicted_2024,
    "actual_vorp_next": y_test_2024.to_numpy(),
    "low_confidence_extreme_delta": add_low_confidence_flag(te.loc[last_fold_te]).to_numpy(),
})
results_2024["error"] = results_2024["predicted_vorp_next"] - results_2024["actual_vorp_next"]
results_2024["abs_error"] = results_2024["error"].abs()
results_2024 = results_2024.sort_values("abs_error").reset_index(drop=True)

mae_check = mean_absolute_error(results_2024["actual_vorp_next"], results_2024["predicted_vorp_next"])
rmse_check = mean_squared_error(results_2024["actual_vorp_next"], results_2024["predicted_vorp_next"]) ** 0.5
rho_check = spearmanr(results_2024["actual_vorp_next"], results_2024["predicted_vorp_next"])[0]
print(f"Reproduced 2024 fold -- MAE {mae_check:.2f}, RMSE {rmse_check:.2f}, Spearman {rho_check:.3f} "
      f"(should match the test_season=2024 row in the constrained fold table above)")

print(f"\nBest 10 predictions (smallest absolute error):")
print(results_2024.head(10).round(1).to_string(index=False))
print(f"\nWorst 10 misses (largest absolute error):")
print(results_2024.tail(10).round(1).to_string(index=False))

Reproduced 2024 fold -- MAE 25.49, RMSE 32.33, Spearman 0.780 (should match the test_season=2024 row in the constrained fold table above)



Best 10 predictions (smallest absolute error):
            player  predicted_vorp_next  actual_vorp_next  low_confidence_extreme_delta  error  abs_error
     Michael Mayer           -54.299999             -54.6                         False    0.3        0.3
       Hunter Long           -82.800003             -84.4                         False    1.5        1.5
      Luke Farrell           -88.500000             -86.9                         False   -1.6        1.6
        Jack Stoll           -95.300003             -97.3                         False    2.0        2.0
     Luke Musgrave           -76.199997             -73.7                         False   -2.5        2.5
     Charlie Kolar           -76.599998             -79.7                         False    3.1        3.1
Darnell Washington           -56.200001             -53.0                         False   -3.2        3.2
     Adam Trautman           -71.900002             -75.4                         False    3.5        3

**Best prediction: Michael Mayer** — predicted -54.3, actual -54.6 (abs error 0.3). An already below-replacement backup role read almost exactly right, the same kind of easy, low-variance case every position's best prediction has landed on.

**Worst miss: Trey McBride** — predicted +61.4, actual +142.0 (abs error 80.6, not flagged: his `vorp_delta_yoy` heading into this prediction was inside the reliable range). The model badly *underestimated* his 2024→2025 breakout into the league's clear TE1. This is directly relevant to the live 2026 section below: McBride is also this year's actual top-VORP TE, and the model has a documented history — within this very notebook — of underselling his continued ascent. That's real, specific evidence to weigh against whatever the model says about his 2026 outlook, not just the generic "flag" boilerplate.

## Live 2026 prediction — a genuine unknown

`te_model.json` — trained on every real season through 2024 — is fed each TE's actual, already-realized 2025 season to predict 2026 VORP, using the top 5 TEs by realized 2025 VORP. Speculative and unverifiable until the season finishes; does not change the validated estimate above (26.33 MAE / 0.689 Spearman).

In [15]:
loaded_te_model = xgb.XGBRegressor()
loaded_te_model.load_model(str(REPO_ROOT / "data" / "models" / "te_model.json"))

top5_2025 = te_2025_features.sort_values("vorp", ascending=False).head(5).copy()
top5_2025["predicted_2026_vorp"] = loaded_te_model.predict(top5_2025[FEATURES])
top5_2025["low_confidence_extreme_delta"] = add_low_confidence_flag(top5_2025)
top5_2025["no_computable_delta"] = top5_2025["vorp_delta_yoy"].isna()

live_predictions = top5_2025[
    ["player_display_name", "vorp", "predicted_2026_vorp", "low_confidence_extreme_delta", "no_computable_delta"] + FEATURES
].rename(columns={"vorp": "actual_2025_vorp"}).reset_index(drop=True)

print("Top 5 TEs by realized 2025 VORP -- live 2026 prediction (SPECULATIVE, unverifiable until the season finishes):\n")
print(live_predictions[["player_display_name", "actual_2025_vorp", "predicted_2026_vorp", "low_confidence_extreme_delta", "no_computable_delta"]]
      .round(1).to_string(index=False))
print("\nFeature rows used (each TE's real, already-realized 2025 season):")
print(live_predictions[["player_display_name"] + FEATURES].round(3).to_string(index=False))

print(f"\nlow_confidence_extreme_delta = True means |vorp_delta_yoy| > {LOW_CONFIDENCE_DELTA_THRESHOLD}.")
print("no_computable_delta = True means this player has no usable prior-season VORP at all (e.g. a true rookie) --")
print("a DIFFERENT, separate reason for caution than the extreme-delta flag, and one the flag itself cannot see")
print("(it silently evaluates to False on a NaN input, which is NOT the same as 'reliable').")

Top 5 TEs by realized 2025 VORP -- live 2026 prediction (SPECULATIVE, unverifiable until the season finishes):


player_display_name  actual_2025_vorp  predicted_2026_vorp  low_confidence_extreme_delta  no_computable_delta
       Trey McBride             142.0            97.599998                         False                False
         Kyle Pitts              55.9            19.799999                         False                False
       Travis Kelce              44.3            42.500000                         False                False
     Dallas Goedert              44.2            15.200000                         False                False
       Tyler Warren              39.6            17.100000                         False                 True

Feature rows used (each TE's real, already-realized 2025 season):
player_display_name  scarcity_z  air_yards_share  receiving_epa  wopr  target_share  draft_tier_Round 2-3  draft_pick_inverse  vorp_delta_yoy  injury_designations_count
       Trey McBride       4.257            0.242         72.716 0.580         0.274                   1.

### Football-sense eyeball pass on the top 5

**Trey McBride** (actual 2025 VORP +142.0 → predicted 2026 +97.6, not flagged: `vorp_delta_yoy` +61.3 sits inside the reliable range): the model calls for a moderate step down from a monster season, but **this exact model already has a documented blind spot for this exact player** — the 2024 fold real-predictions check above shows McBride as the single worst miss in the whole notebook, underestimating his 2024→2025 breakout by 80.6 points. That history doesn't prove 2026 will be underestimated again, but it's concrete, specific, in-notebook evidence that this model's error mode on McBride specifically has been to undersell his continued ascent, not oversell it. **Worth reading his predicted decline with real skepticism, grounded in a repeat pattern rather than a generic caveat.**

**Kyle Pitts** (+55.9 → +19.8, not flagged): real-world context matters here and argues against the model's aggressive discount. Pitts was the subject of extensive trade speculation through the 2025 offseason but ultimately stayed in Atlanta and signed a three-year, $54M extension after his best season yet (career highs in catches, first downs, and touchdowns) ([Yahoo Sports](https://sports.yahoo.com/articles/kyle-pitts-trade-sweepstakes-heating-163627109.html), [SI](https://www.si.com/nfl/falcons/onsi/espn-stokes-the-fire-of-kyle-pitts-trade-talk-for-falcons)). A resolved trade situation plus a new long-term contract is a real stability signal a backward-looking feature set can't directly encode — the model's steep discount may be reading a real recent step-up as noise. **Update (via `08d_shap_te.ipynb`'s SHAP analysis): this trade-context caveat was incomplete on its own.** SHAP shows a second, real, measurable driver also pulling his prediction down that this paragraph originally didn't mention: his `injury_designations_count` (2.0 that season) contributes -5.9, the second-largest non-`scarcity_z` factor in his row, ahead of `receiving_epa` and `wopr`. The trade/contract stability argument above still stands as a real, backward-looking-feature-set blind spot, but it isn't the only relevant factor behind his actual predicted number. **Worth skepticism on the magnitude, for a different reason than the standard extreme-delta caveat.**

**Travis Kelce** (+44.3 → +42.5, not flagged: `vorp_delta_yoy` +5.0, essentially flat): real-world reporting is more pessimistic than this near-flat prediction suggests — at 36 (37 in October), analysts have described each of his last two seasons as "the worst of his career" and noted he's "clearly his age" ([Yahoo Sports](https://sports.yahoo.com/articles/travis-kelce-retire-heres-chiefs-202856531.html)). **A structural reason this model can't see that risk clearly: TE's own locked 9-feature set is the only position of the four that does NOT include `age` at all** (RFECV eliminated it for TE, unlike QB/RB/WR). A near-flat prediction for a 37-year-old with a publicly documented decline may be more optimistic than the real aging risk warrants — not because of the extreme-delta mechanism, but because of what TE's feature set structurally cannot represent.

**Dallas Goedert** (+44.2 → +15.2, not flagged: `vorp_delta_yoy` +69.2, `injury_designations_count` 2.0): real-world context shows a more injury-disrupted season than the modest count suggests — Goedert dealt with a knee injury (Week 2), a hamstring injury (Weeks 7-10), and a PCL injury (Weeks 14-17), yet still posted a near career-best season when actually on the field (60 catches, career-high 11 TDs) ([Yahoo Sports](https://sports.yahoo.com/article/eagles-injury-report-dallas-goedert-145141362.html)). `injury_designations_count` (a weekly Q/D/O tally) is a real but blunt proxy — it doesn't distinguish "tagged Questionable and played" from "missed four full weeks," so a season with three distinct injury absences can still show a modest count. The steep predicted decline may be capturing real recurring-injury risk this feature under-measures, or may just be ordinary regression from a strong per-game rate — genuinely ambiguous either way.

**Tyler Warren** (+39.6 → +17.1, `no_computable_delta` = True, mechanically NOT flagged by `low_confidence_extreme_delta` even though this is arguably the least reliable row of the five): a true rookie (Colts, No. 14 overall in the 2025 draft) who had a genuinely elite debut — a franchise rookie-TE receiving-yardage record, a Pro Bowl nod, and top-6 among *all* NFL tight ends (not just rookies) in both receptions and receiving yards ([Colts.com](https://www.colts.com/news/2025-colts-rookie-review-tyler-warren-tight-end-offense-versatility-growth)). With no prior season, `vorp_delta_yoy` is `NaN` and the flag silently reads `False` — this notebook's own flag mechanism cannot distinguish "reliable" from "no history to judge reliability at all." The predicted decline for a clearly ascending, no-red-flags rookie is worth real skepticism, and the *reason* for that skepticism (missing history, not an extreme measured delta) is a genuinely different failure mode than every other flagged case across all four positions so far. **Confirmed by SHAP (`08d_shap_te.ipynb`): the model does not simply treat a missing `vorp_delta_yoy` as zero or ignore it** — XGBoost routes `NaN` through its own dedicated missing-value branch, and for Warren's row that branch contributes +8.1, a *larger* magnitude than any of the other three flagged players' actual, measured `vorp_delta_yoy` contributions (-0.2 to -3.0). Concrete evidence the model genuinely treats "no history" as its own distinct case rather than folding it into ordinary delta behavior — real support for keeping `no_delta_history` a separate flag from `low_confidence_extreme_delta`, the design decision made when this bug was first fixed (see `roadmap.md`).

**Bottom line on this pass**: none of the top 5 are caught by `low_confidence_extreme_delta` this time (TE's naturally smaller deltas among established starters plus one true rookie with no delta at all), but 4 of the 5 still warrant real, specific skepticism for reasons the flag doesn't capture: McBride (documented model blind spot for this specific player), Pitts (resolved trade + new contract as a stability signal), Kelce (feature set structurally excludes `age`, the one input that would price his real aging risk), and Warren (missing history that the flag mechanically can't see). This is a different, arguably more informative kind of scrutiny than the extreme-delta mechanism alone would have surfaced — the flag is necessary but not sufficient for a full football-sense read.

Sources:
- [Kyle Pitts trade sweepstakes heating up and these 4 teams are the favorites to land him](https://sports.yahoo.com/articles/kyle-pitts-trade-sweepstakes-heating-163627109.html)
- [ESPN Stokes the Fire of Kyle Pitts Trade Talk for Falcons](https://www.si.com/nfl/falcons/onsi/espn-stokes-the-fire-of-kyle-pitts-trade-talk-for-falcons)
- [Eagles injury report: Dallas Goedert misses practice with knee injury](https://sports.yahoo.com/article/eagles-injury-report-dallas-goedert-145141362.html)
- [2025 Colts Rookie Review: Tyler Warren shows off his versatility and availability](https://www.colts.com/news/2025-colts-rookie-review-tyler-warren-tight-end-offense-versatility-growth)
- [When will Travis Kelce retire? Here's what the Chiefs TE has said](https://sports.yahoo.com/articles/travis-kelce-retire-heres-chiefs-202856531.html)

## Honest verdict

**TE is the first genuine tie in the monotonic-constraint diagnostic.** Constrained wins MAE (26.33 vs. 26.59); unconstrained wins Spearman by a noise-level 0.002. No prior position (QB, RB, WR) had this problem — each won cleanly on both metrics for one configuration. TE's tie is broken toward the constrained model because MAE is what this notebook's own tuning objective directly optimizes; this is a defensible but explicit judgment call, not a repeat of a clean prior result.

| Model | MAE | Spearman |
|---|---|---|
| Naive (this season's VORP) | 27.28 | 0.691 |
| Phase 4 tuned, no constraints (diagnostic) | 26.59 | 0.691 |
| **Phase 4 tuned + monotonic (official, shipped)** | **26.33** | **0.689** |

**The naive-baseline comparison is the weakest of any position so far**: the official model's Spearman (0.689) doesn't even clear naive's (0.691) — a small give-back, not an improvement. MAE improves a real but modest 3.5%. TE's edge over "assume next season repeats this season" is about average error size, not about getting the ranking more right.

**`vorp_delta_yoy`'s extreme-tail weakness is now 4-for-4 across positions, and TE's is the sharpest yet**: 1.66x degradation (QB 1.29x → RB 1.41x → WR 1.45x → TE 1.66x), with a clean monotonic quartile progression (20.76 → 21.67 → 30.79 → 36.30). `low_confidence_extreme_delta` is built into every prediction table, flagging 2.7% of training rows — the lowest flagged share of any position, consistent with TE's naturally smaller VORP swings.

**The `scarcity_z` tail check now looks like a universal pattern, not a position-family split.** TE joins WR in showing the top decile (the actual TE1s, MAE 44.82) as the hardest segment in the dataset — worse even than the delta extreme-tail bucket — while the bottom decile (MAE 14.97) is easy, same as everywhere else. **This section originally read "QB and RB show the opposite pattern," framing it as a 2-and-2 split between receiving-volume and run/pocket-presence positions — that framing has since been corrected.** `06b_model_rb.ipynb` was given the same decile-separated check (it had none before) and found RB's own top decile is *also* its hardest segment; RB's previously-cited "fits better" number was an average of both deciles that happened to land near neutral, masking the same asymmetry. QB's version of this claim has still not been checked with real, committed notebook code, and should be treated as open rather than assumed to hold just because it was written down first.

**`draft_tier_Round 2-3` survives real tuning, but only in the model that actually shipped**: it holds essentially Phase 3's original rank (6th of 9, vs. 6th of 10 untuned) in the official constrained model, but falls to dead last under the unconstrained alternative this notebook seriously considered. A real, if narrower, survival than QB's TD-rate regression (which RFECV rejected outright in every configuration) — genuine signal, but more fragile to the constraint choice than any other secondary feature checked across all four positions.

**Real-predictions and live-2026 checks surface a genuinely different kind of finding than QB/RB/WR's**: none of TE's top-5 realized-2025 players trip the extreme-delta flag, but 4 of 5 warrant real skepticism for reasons the flag can't see — Trey McBride (this exact model's own documented blind spot, having underestimated his prior breakout by 80.6 points in-notebook), Kyle Pitts (a resolved trade situation and new contract the model can't encode), Travis Kelce (TE's feature set is the only one of the four that excludes `age` entirely, so real, well-documented aging risk at 36 has no direct input to load onto), and Tyler Warren (a true rookie whose missing `vorp_delta_yoy` makes the flag mechanically silent, not reliable). The flag mechanism, while real and load-bearing, is not sufficient on its own for TE's specific mix of players this year.

**Bottom line**: TE ships as a real but weaker model than QB/RB/WR — its constraint choice is a genuine coin-flip resolved by explicit tiebreak rather than a clean win, and it's the first position whose tuned Spearman doesn't clear naive. It reconfirms and sharpens the `vorp_delta_yoy` extreme-tail pattern for a fourth straight position, and its `scarcity_z` top-decile weakness (now shared with WR and RB once all three were checked the same way) looks like a real, near-universal property of this feature rather than a position-specific one — pending QB's own real check. `te_model.json` and `fold_metrics_te.csv` reflect the constrained, official model.